In [6]:
"""
Self-Pruning Neural Network for CIFAR-10 — FINAL VERSION
=========================================================
Method: Sigmoid gates + Straight-Through Estimator (STE) hard mask
- Forward pass  : uses HARD binary mask (gate > threshold → 1, else → 0)
- Backward pass : gradients flow through sigmoid (STE trick)
- This guarantees TRUE structural zeros from early in training
- L1 on sigmoid gates pushes scores toward -inf → gate = 0 in mask
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os, time, json

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE    = 128
EPOCHS        = 30
LR_MAIN       = 1e-3
LR_GATE       = 5e-3
WEIGHT_DECAY  = 1e-4
LAMBDAS       = [1e-3, 1e-2, 5e-2]
PRUNE_THRESH  = 0.5    # sigmoid(gate_score) < 0.5 → gate_score < 0 → hard mask = 0
SAVE_DIR      = "output"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Device : {DEVICE}")
print(f"Epochs : {EPOCHS}  |  Lambdas : {LAMBDAS}")


class STEFunction(torch.autograd.Function):
    """
    Straight-Through Estimator (STE).
    Forward  : hard binary threshold (exact 0 or 1)
    Backward : gradient passes through as if the function were identity
    This is the standard trick used in BinaryConnect, ProxQuant, etc.
    """
    @staticmethod
    def forward(ctx, gates, threshold=0.5):
        return (gates >= threshold).float()

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None   # pass gradient straight through


def ste_threshold(gates, threshold=0.5):
    return STEFunction.apply(gates, threshold)


class PrunableLinear(nn.Module):
    """
    Linear layer with Sigmoid + STE hard mask.
    - gate_scores are learnable parameters
    - gates = sigmoid(gate_scores)  [soft, for L1 loss + gradient]
    - mask  = STE(gates)            [hard 0/1, for actual weight masking]
    - Forward uses mask (real zeros), Backward uses sigmoid gradient (real updates)
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.weight      = nn.Parameter(torch.empty(out_features, in_features))
        self.bias_param  = nn.Parameter(torch.zeros(out_features)) if bias else None
        self.gate_scores = nn.Parameter(torch.empty(out_features, in_features))
        self._init_parameters()

    def _init_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=0.01)
        # Init near 0 → sigmoid ≈ 0.5 → right at the decision boundary
        # L1 will immediately start pushing scores negative → mask becomes 0
        nn.init.normal_(self.gate_scores, mean=0.0, std=0.01)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gates = torch.sigmoid(self.gate_scores)          # soft gates ∈ (0,1)
        if self.training:
            mask = ste_threshold(gates, threshold=0.5)   # hard binary mask, STE backward
        else:
            mask = (gates >= 0.5).float()                # hard binary at eval too
        pruned_weights = self.weight * mask
        return F.linear(x, pruned_weights, self.bias_param)

    def get_soft_gates(self) -> torch.Tensor:
        """Soft gates with grad — for L1 sparsity loss."""
        return torch.sigmoid(self.gate_scores)

    def get_hard_mask(self) -> torch.Tensor:
        """Hard binary mask without grad — for sparsity reporting."""
        with torch.no_grad():
            return (torch.sigmoid(self.gate_scores) >= 0.5).float()

    def sparsity(self) -> float:
        """Fraction of weights that are hard-masked to 0."""
        mask = self.get_hard_mask()
        return (mask == 0).float().mean().item()


class SelfPruningNet(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.20),

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            PrunableLinear(256 * 4 * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            PrunableLinear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

    def prunable_layers(self):
        return [m for m in self.modules() if isinstance(m, PrunableLinear)]

    def sparsity_loss(self):
        """
        L1 on soft sigmoid gates (with grad).
        Pushes gate_scores negative → sigmoid → 0 → hard mask becomes 0.
        """
        all_gates = torch.cat([
            layer.get_soft_gates().view(-1)
            for layer in self.prunable_layers()
        ])
        return all_gates.mean()

    def overall_sparsity(self) -> float:
        """% of weights actually masked to 0 by hard binary mask."""
        total, pruned = 0, 0
        for layer in self.prunable_layers():
            mask = layer.get_hard_mask()
            pruned += (mask == 0).sum().item()
            total  += mask.numel()
        return 100.0 * pruned / total


def get_loaders():
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2470, 0.2435, 0.2616)
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    train_ds = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, test_loader


def total_loss(logits, targets, model, lam, smoothing=0.1):
    log_probs   = F.log_softmax(logits, dim=1)
    smooth_loss = -log_probs.mean(dim=1).mean()
    true_loss   = F.nll_loss(log_probs, targets)
    cls_loss    = (1 - smoothing) * true_loss + smoothing * smooth_loss
    sp_loss     = model.sparsity_loss()
    return cls_loss + lam * sp_loss


def make_optimizers(model):
    gate_params, other_params = [], []
    for name, param in model.named_parameters():
        if "gate_scores" in name:
            gate_params.append(param)
        else:
            other_params.append(param)
    optimizer = optim.AdamW([
        {"params": other_params, "lr": LR_MAIN, "weight_decay": WEIGHT_DECAY},
        {"params": gate_params,  "lr": LR_GATE, "weight_decay": 0.0},
    ])
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    return optimizer, scheduler


def train_epoch(model, loader, optimizer, lam):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = total_loss(logits, labels, model, lam)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct      += logits.argmax(1).eq(labels).sum().item()
        total        += imgs.size(0)
    return running_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        correct += model(imgs).argmax(1).eq(labels).sum().item()
        total   += imgs.size(0)
    return 100.0 * correct / total


def run_experiment(lam, train_loader, test_loader):
    print(f"\n{'='*60}")
    print(f"  Lambda = {lam}")
    print(f"{'='*60}")

    model                = SelfPruningNet().to(DEVICE)
    optimizer, scheduler = make_optimizers(model)
    history = {"train_acc": [], "test_acc": [], "sparsity": [], "loss": []}

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, lam)
        te_acc          = evaluate(model, test_loader)
        sparsity        = model.overall_sparsity()
        scheduler.step()

        history["train_acc"].append(tr_acc)
        history["test_acc"].append(te_acc)
        history["sparsity"].append(sparsity)
        history["loss"].append(tr_loss)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:>2d}/{EPOCHS} | Loss {tr_loss:.4f} | "
                  f"Train {tr_acc:.1f}% | Test {te_acc:.1f}% | "
                  f"Sparsity {sparsity:.1f}% | {time.time()-t0:.1f}s")

    final_acc      = evaluate(model, test_loader)
    final_sparsity = model.overall_sparsity()
    print(f"\n  Final → Test Accuracy: {final_acc:.2f}%  |  Sparsity: {final_sparsity:.2f}%")

    all_gates = torch.cat([
        layer.get_soft_gates().detach().view(-1).cpu()
        for layer in model.prunable_layers()
    ]).numpy()

    return {
        "lambda": lam,
        "test_accuracy": round(final_acc, 2),
        "sparsity": round(final_sparsity, 2),
        "history": history,
        "gates": all_gates,
    }


def save_plots(results):
    colors = ["#2196F3", "#FF9800", "#E91E63"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for i, res in enumerate(results):
        ep = range(1, EPOCHS + 1)
        axes[0].plot(ep, res["history"]["test_acc"], color=colors[i], lw=2, label=f"λ={res['lambda']}")
        axes[1].plot(ep, res["history"]["sparsity"], color=colors[i], lw=2, label=f"λ={res['lambda']}")
        axes[2].plot(ep, res["history"]["loss"],     color=colors[i], lw=2, label=f"λ={res['lambda']}")
    for ax, title, ylabel in zip(
        axes,
        ["Test Accuracy over Epochs", "Sparsity (%) over Epochs", "Training Loss over Epochs"],
        ["Test Accuracy (%)", "Sparsity (%)", "Loss"]
    ):
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.suptitle("Self-Pruning NN — CIFAR-10 (STE Gates)", fontsize=15, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/training_curves.png", dpi=150, bbox_inches="tight")
    plt.close()

    fig, axes = plt.subplots(1, len(results), figsize=(6*len(results), 5))
    for ax, res in zip(axes, results):
        ax.hist(res["gates"], bins=80, color="#1976D2", edgecolor="none", alpha=0.85)
        ax.axvline(0.5, color="red", linestyle="--", lw=1.5, label="threshold=0.5")
        ax.set_title(f"λ={res['lambda']}\nAcc={res['test_accuracy']}%  Sparsity={res['sparsity']}%",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("Soft Gate Value (sigmoid)"); ax.set_ylabel("Count")
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.suptitle("Gate Distributions — STE Pruning", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/gate_distributions.png", dpi=150, bbox_inches="tight")
    plt.close()

    lam_labels = [str(r["lambda"]) for r in results]
    accs  = [r["test_accuracy"] for r in results]
    spars = [r["sparsity"]      for r in results]
    x, width = np.arange(len(lam_labels)), 0.35
    fig, ax1 = plt.subplots(figsize=(9, 6))
    ax2 = ax1.twinx()
    bars1 = ax1.bar(x - width/2, accs,  width, label="Test Accuracy (%)", color="#1976D2", alpha=0.85)
    bars2 = ax2.bar(x + width/2, spars, width, label="Sparsity (%)",       color="#E91E63", alpha=0.85)
    ax1.set_xlabel("Lambda (λ)", fontsize=12)
    ax1.set_ylabel("Test Accuracy (%)", color="#1976D2", fontsize=12)
    ax2.set_ylabel("Sparsity (%)",      color="#E91E63", fontsize=12)
    ax1.set_xticks(x); ax1.set_xticklabels(lam_labels)
    ax1.set_ylim(0, 105); ax2.set_ylim(0, 105)
    for bar in bars1:
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=10)
    for bar in bars2:
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=10)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1+lines2, labels1+labels2, loc="upper left")
    ax1.set_title("Accuracy vs Sparsity Trade-off", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/lambda_tradeoff.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Plots saved to ./{SAVE_DIR}/")


if __name__ == "__main__":
    train_loader, test_loader = get_loaders()
    results = []

    for lam in LAMBDAS:
        res = run_experiment(lam, train_loader, test_loader)
        results.append(res)

    summary = [{"lambda": r["lambda"],
                "test_accuracy_%": r["test_accuracy"],
                "sparsity_%": r["sparsity"]} for r in results]
    with open(f"{SAVE_DIR}/results_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\n\n" + "="*60)
    print("FINAL RESULTS SUMMARY")
    print("="*60)
    print(f"{'Lambda':<12} {'Test Acc (%)':>14} {'Sparsity (%)':>14}")
    print("-"*40)
    for s in summary:
        print(f"{s['lambda']:<12} {s['test_accuracy_%']:>14.2f} {s['sparsity_%']:>14.2f}")

    save_plots(results)
    print("\nAll done!")

Device : cuda
Epochs : 30  |  Lambdas : [0.001, 0.01, 0.05]

  Lambda = 0.001
  Epoch  1/30 | Loss 1.7967 | Train 40.7% | Test 57.3% | Sparsity 61.5% | 23.0s
  Epoch  5/30 | Loss 1.1856 | Train 70.2% | Test 77.7% | Sparsity 70.2% | 23.1s
  Epoch 10/30 | Loss 1.0189 | Train 78.3% | Test 83.9% | Sparsity 71.9% | 22.9s
  Epoch 15/30 | Loss 0.9266 | Train 82.5% | Test 85.9% | Sparsity 71.9% | 23.1s
  Epoch 20/30 | Loss 0.8662 | Train 85.4% | Test 87.9% | Sparsity 71.4% | 23.2s
  Epoch 25/30 | Loss 0.8291 | Train 87.0% | Test 89.3% | Sparsity 71.1% | 23.5s
  Epoch 30/30 | Loss 0.8169 | Train 87.7% | Test 89.5% | Sparsity 71.0% | 23.5s

  Final → Test Accuracy: 89.53%  |  Sparsity: 71.03%

  Lambda = 0.01
  Epoch  1/30 | Loss 1.8257 | Train 39.7% | Test 53.5% | Sparsity 62.7% | 23.2s
  Epoch  5/30 | Loss 1.2079 | Train 69.4% | Test 78.0% | Sparsity 72.9% | 23.2s
  Epoch 10/30 | Loss 1.0310 | Train 77.7% | Test 84.0% | Sparsity 75.9% | 23.5s
  Epoch 15/30 | Loss 0.9372 | Train 82.3% | Test 87